#Assignment - Ensemble Learning

####**Question 1: What is Ensemble Learning in machine learning? Explain the key idea behind it.**

**Soln-** Ensemble learning is a machine learning technique that combines predictions from multiple individual models (often called "weak learners") to create a single, superior model. By aggregating these predictions via voting or averaging ensemble methods achieve higher predictive accuracy, improved robustness, and reduced overfitting compared to any single model.

The core philosophy is that a group of diverse, moderately accurate models can "team up" to produce a better outcome than any single high-performance model. It functions on the "wisdom of the crowd" principle.

**Mitigating Weaknesses:** It reduces variance, bias, or overfitting by allowing models to correct each other's mistakes.

**Diverse Perspectives:** By training models on different subsets of data (bagging) or sequentially focusing on errors (boosting), the ensemble covers a broader range of patterns.

####**Question 2: What is the difference between Bagging and Boosting?**

**Soln-** Bagging and Boosting are ensemble techniques to improve model performance, but they differ fundamentally: Bagging reduces variance (overfitting) by training independent, parallel models on random data subsets, while Boosting reduces bias (underfitting) by training models sequentially, with each new model fixing the errors of the previous one. Bagging (e.g., Random Forest) is robust against noise, while Boosting (e.g., XGBoost) is better for maximizing accuracy

####**Question 3: What is bootstrap sampling and what role does it play in Bagging methods like Random Forest?**

**Soln-** Bootstrap sampling is a statistical resampling technique that involves repeatedly drawing random samples from a dataset with replacement. In this process, every observation has an equal chance of being selected for a new sample, which can result in some data points appearing multiple times while others are omitted entirely.

* Diverse Training Sets: It creates multiple slightly different subsets of the
original training data. Each subset is used to train an independent base learner, such as a Decision Tree.

* Variance Reduction: By training models on these diverse samples and averaging their results (for regression) or using majority voting (for classification), bagging significantly reduces the overall model variance. This helps prevent the ensemble from overfitting to specific quirks or noise in the original dataset.

* De-correlation (Random Forest): In Random Forest, bootstrap sampling works alongside feature randomness (selecting a subset of features for each split) to further decorrelate individual trees, making the final aggregate prediction even more robust.

* Out-of-Bag (OOB) Evaluation: On average, a bootstrap sample only contains about 63.2% of the unique original observations. The remaining ~36.8% of "unsampled" data points (called Out-of-Bag samples) can be used as a built-in validation set to estimate the model’s performance without needing a separate test set





####**Question 4: What are Out-of-Bag (OOB) samples and how is OOB score used to evaluate ensemble models?**

**Soln-** In bagging, each model is trained on a bootstrap sample (random subset with replacement). OOB samples are the data rows not included in that subset.

The OOB score evaluates the ensemble by aggregating predictions from these "unseen" samples, providing an unbiased, internal validation metric without requiring a separate test set

####**Question 5: Compare feature importance analysis in a single Decision Tree vs. a Random Forest.**


**Soln-**  
*   A single tree's importance is erratic and changes significantly with new data. Random Forest provides a more stable, balanced ranking due to ensemble averaging.
*   Single trees may overfit and give high importance to noise features. Random Forest reduces this variance, ensuring feature importance scores are more representative of the data.
* A single tree often arbitrarily selects one feature from a correlated group and ignores the rest. Random Forest tends to share importance across correlated features, giving a better picture of predictive power.
* Single trees are easy to visualize to understand why a feature is important. Random Forest is a "black box," but its feature importance rankings are more reliable for feature selection.
* Single trees calculate importance based on local splitting decisions (greedy). Random Forests calculate importance by averaging the total reduction in impurity (MDI) caused by a feature across all trees.



####**6. Write a Python program to:**
* Load the Breast Cancer dataset using
sklearn.datasets.load_breast_cancer()
* Train a Random Forest Classifier
* *italicized text* Print the top 5 most important features based on feature importance scores.

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train, y_train)

importances = rf_clf.feature_importances_
feature_imp_df = pd.DataFrame(
    {'Feature': cancer.feature_names, 'Importance': importances}
).sort_values(by='Importance', ascending=False)

print("Top 5 Most Important Features:")
print(feature_imp_df.head(5).to_string(index=False))


Top 5 Most Important Features:
             Feature  Importance
 mean concave points    0.141934
worst concave points    0.127136
          worst area    0.118217
      mean concavity    0.080557
        worst radius    0.077975


####**Question 7: Write a Python program to:**
* Train a Bagging Classifier using Decision Trees on the Iris dataset
* Evaluate its accuracy and compare with a single Decision Tree

In [2]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score

data = load_iris()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

bagging_model = BaggingClassifier(estimator=DecisionTreeClassifier(),
                                   n_estimators=100,
                                   random_state=42)
bagging_model.fit(X_train, y_train)

dt_accuracy = accuracy_score(y_test, dt_model.predict(X_test))
bagging_accuracy = accuracy_score(y_test, bagging_model.predict(X_test))

print(f"Accuracy of a single Decision Tree: {dt_accuracy:.4f}")
print(f"Accuracy of Bagging Classifier: {bagging_accuracy:.4f}")
print(f"Improvement: {bagging_accuracy - dt_accuracy:.4f}")


Accuracy of a single Decision Tree: 1.0000
Accuracy of Bagging Classifier: 1.0000
Improvement: 0.0000


####**Question 8: Write a Python program to:**
* Train a Random Forest Classifier
* Tune hyperparameters max_depth and n_estimators using GridSearchCV
* Print the best parameters and final accuracy

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

data = load_iris()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10, 20]
}

grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, scoring='accuracy')

grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
final_accuracy = accuracy_score(y_test, y_pred)

print(f"Final Test Accuracy: {final_accuracy:.4f}")


Best Parameters: {'max_depth': None, 'n_estimators': 200}
Final Test Accuracy: 1.0000


####**Question 9: Write a Python program to:**
* Train a Bagging Regressor and a Random Forest Regressor on the California
Housing dataset
* Compare their Mean Squared Errors (MSE)

In [4]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

housing = fetch_california_housing()
X, y = housing.data, housing.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

bagging_reg = BaggingRegressor(
    estimator=DecisionTreeRegressor(),
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
bagging_reg.fit(X_train, y_train)
bagging_pred = bagging_reg.predict(X_test)
bagging_mse = mean_squared_error(y_test, bagging_pred)

rf_reg = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf_reg.fit(X_train, y_train)
rf_pred = rf_reg.predict(X_test)
rf_mse = mean_squared_error(y_test, rf_pred)

print(f"Bagging Regressor (DecisionTree base) MSE: {bagging_mse:.4f}")
print(f"Random Forest Regressor MSE:             {rf_mse:.4f}")

if rf_mse < bagging_mse:
    print("\nRandom Forest performed better (lower MSE).")
else:
    print("\nBagging Regressor performed better (lower MSE).")


Bagging Regressor (DecisionTree base) MSE: 0.2559
Random Forest Regressor MSE:             0.2554

Random Forest performed better (lower MSE).


####**Question 10: You are working as a data scientist at a financial institution to predict loan default. You have access to customer demographic and transaction history data. You decide to use ensemble techniques to increase model performance.**
Explain your step-by-step approach to:
* Choose between Bagging or Boosting
* Handle overfitting
* Select base models
* Evaluate performance using cross-validation
* Justify how ensemble learning improves decision-making in this real-world
context.


##1. Choosing Between Bagging and Boosting

In a financial context like loan default, the choice depends on the nature of the data and the business objective:

* **Boosting (Recommended for Accuracy):** Algorithms like **XGBoost, LightGBM, or CatBoost** are generally preferred for loan defaults. These models are designed to minimize bias and are highly effective at capturing complex, non-linear relationships in tabular data (like transaction history). They sequentially focus on "hard-to-classify" customers who are likely to default.
* **Bagging (Recommended for Stability):** If the dataset is very noisy or small, **Random Forest** (Bagging) is better. It reduces variance and is less likely to overfit to outliers compared to Boosting.



## 2. Selecting Base Models
The "base models" are the individual learners that make up the ensemble:

* **Decision Trees:** These are the most common base learners for both Bagging and Boosting because they can handle categorical data (demographics) and numerical data (transaction amounts) without extensive scaling.
* **Diversity is Key:** For a **Voting/Stacking** ensemble, I might combine diverse models like a Logistic Regression (to capture linear trends), a Decision Tree (for non-linearities), and a k-Nearest Neighbor (for local patterns). Diversity ensures that when one model makes a mistake, others can correct it.



## 3. Handling Overfitting
Since loan datasets often have many features and a class imbalance (fewer defaulters than non-defaulters), overfitting is a major risk. I would:

* **Pruning & Constraints:** Limit the `max_depth` of trees and increase `min_samples_leaf` to prevent models from memorizing specific transactions.
* **Regularization:** In Boosting, use $L1$ (Lasso) or $L2$ (Ridge) regularization parameters to penalize overly complex models.
* **Subsampling:** Use **Stochastic Gradient Boosting** (sampling rows) and **Feature Subsampling** (sampling columns) to ensure no single tree relies too heavily on a specific "noisy" feature.



## 4. Evaluating Performance using Cross-Validation
Standard accuracy is misleading in finance because most people do not default. I would use:

* **Stratified K-Fold Cross-Validation:** This ensures that each "fold" of the data contains the same proportion of defaulters as the original dataset, providing a stable estimate of performance.
* **Relevant Metrics:** Instead of accuracy, I would evaluate the **AUC-ROC** (to measure the ability to distinguish between classes) and **Precision-Recall curves** (since the cost of missing a defaulter is much higher than the cost of a false alarm).


## 5. Justifying Ensemble Learning in a Real-World Context
Ensemble learning significantly improves financial decision-making because:

* **Risk Mitigation:** Financial institutions prioritize "robustness." A single model might fail if consumer behavior shifts slightly, but an ensemble averages out these errors, leading to more stable and reliable risk assessments.
* **Capturing Complex Patterns:** Loan default is rarely caused by one factor. It’s usually a combination of demographic stability and transaction volatility. Ensembles are mathematically better at identifying these multi-dimensional interactions.
* **Reduced Bias:** By combining multiple perspectives, ensembles reduce the "model bias" that might occur if a single algorithm is poorly suited for a specific segment of the population.